# T18 -- Lung-Region Attention Module (Candidate A)
## DenseNet121 + supervised spatial attention, trained on the COVID-19 Radiography Database

**Owner:** Member 1 | **Week 2, Milestone M2** | Novel contribution (Candidate A of 3)

**Design doc:** `Claude Working Files/T18_Lung_Region_Attention_WBS.md` -- read sections 1-7
before touching this notebook; this file follows that design exactly, section by section
(S0-S16). Where this notebook and `Member1_Guide.md` disagree, the WBS wins (see WBS section 4).

**Platform:** Kaggle, T4 (confirmed in S0 -- no sm_60 / P100 compatibility pin needed).

**S0 pre-flight -- resolved decisions (see WBS section 12.5 / chat log for full reasoning):**
- Lung masks: the COVID-19 Radiography Dataset's bundled `masks/` folder (confirmed official,
  not a fallback -- same folder structure locally and on the Kaggle-hosted dataset).
- Faithfulness (Grad-CAM EIL) definition: Member 5's T30 definition is not yet in the repo
  (checked `notebooks/candidate-c-grad-cam-shortcut-suppression-loss.ipynb` -- it has a
  training-time suppression loss on raw activations, not a post-hoc scoring function). Using
  our own documented definition (WBS section 6.3); swap later if T30's differs -- it's a
  metric function, not a training-time dependency.
- Hyperparameters: T13 (Member 2's DenseNet121 HP tuning) has no committed winner --
  `notebooks/baseline-cnn-model-dnn-research.ipynb` only exposes phase1_lr/phase2_lr as
  argparse *defaults* (1e-3/1e-5), it never sweeps them. Adopting T16's measured winner
  instead (WBS section 3.4a): `phase1_lr=3e-4, phase2_lr=3e-5, weight_decay=1e-3`.
- Mask-provenance wording (proposal says "automatically generated"; we use the dataset's
  supplied masks) recorded as a known discrepancy for the module card -- not blocking.


## Environment check

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    major, minor = torch.cuda.get_device_capability(0)
    print(f"Compute capability: sm_{major}{minor}")
    if (major, minor) == (6, 0):
        print("WARNING: Tesla P100 (sm_60) detected -- the AuxSeg notebook's torch==2.8.0+cu126 "
              "pin is required before any further torch import. This notebook was designed for "
              "T4 (S0) and does NOT include that pin by default.")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## S1 -- Config

Loads `configs/densenet121_lung_attention.yaml` (arm A2's config). Other arms are produced
from this base config via `merge_overrides()` at the point each arm is trained (S8/S9/S10) --
see the WBS's arm-to-flags table (Appendix A.3) for the exact overrides per arm.


In [ ]:
import sys
from pathlib import Path

# On Kaggle the repo isn't on sys.path by default. Upload the repo as a Kaggle Dataset
# (or clone it in a setup cell) and adjust this path, or run this notebook from a local
# clone where the repo root is already the working directory.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    # Kaggle fallback -- adjust to wherever the repo dataset/clone actually lands.
    REPO_ROOT = Path("/kaggle/working/Chest-X-ray-Disease-Detection")
sys.path.insert(0, str(REPO_ROOT))

from src.utils import load_config, merge_overrides

cfg = load_config(REPO_ROOT / "configs" / "densenet121_lung_attention.yaml")
print("Loaded config for experiment:", cfg["experiment"]["name"])
import json
print(json.dumps(cfg, indent=2))


## S2 -- Data layer -- mask-paired dataset + verification

Copy `JointTransform`, `CXRWithMaskDataset`, `stratified_split`, `build_dataloaders`, `compute_class_weights` from the AuxSeg notebook verbatim, then add reproducible worker seeding (WBS section 4.8). Run the Appendix A.1 verification + alignment-eyeball cells before proceeding -- a misaligned mask silently invalidates every downstream number.

This is also where `artifacts/splits/split_manifest_v1.csv` gets emitted (S1 step 5 of the WBS) using the real dataset.

In [ ]:
from pathlib import Path
from src.datasets import build_dataloaders

# COVID-19 Radiography Dataset -- local path (this repo) vs Kaggle-hosted copy.
# S0 decision: platform is Kaggle/T4; masks are the dataset's bundled masks/ folder
# (both paths below ship the same masks/ folder structure).
_LOCAL_DATA_DIR = "/Volumes/My Disk 2/My Projects/Chest Disease Detection/COVID-19_Radiography_Dataset"
_KAGGLE_DATA_DIR = "/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset"
DATA_DIR = _LOCAL_DATA_DIR if Path(_LOCAL_DATA_DIR).exists() else _KAGGLE_DATA_DIR
print("Using DATA_DIR:", DATA_DIR)

SPLIT_MANIFEST = REPO_ROOT / "artifacts" / "splits" / "split_manifest_v1.csv"
assert SPLIT_MANIFEST.exists(), (
    f"{SPLIT_MANIFEST} not found -- it must be committed by S1, not regenerated here "
    "(docs/experiment_policy.md 'Dataset Split')."
)

train_loader, val_loader, test_loader, class_names, train_targets, datasets = build_dataloaders(
    DATA_DIR,
    img_size=cfg["dataset"]["image_size"],
    batch_size=cfg["training"]["batch_size"],
    seed=cfg["experiment"]["seed"],
    num_workers=4,
    split_manifest_path=SPLIT_MANIFEST,
)
print(f"Classes ({len(class_names)}): {class_names}")
print(f"Train/Val/Test sizes: {len(train_loader.dataset)}/{len(val_loader.dataset)}/{len(test_loader.dataset)}")

import torch.nn as nn
from src.datasets import compute_class_weights

class_weights = compute_class_weights(train_targets, num_classes=len(class_names)).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
print("Class weights:", class_weights.cpu().tolist())


In [ ]:
# --- S2 verification cell (WBS Appendix A.1) ---
import numpy as np
import torch
from PIL import Image

CLASSES = ["COVID", "Lung_Opacity", "Normal", "Viral Pneumonia"]

# 1) split sizes + class order (also implicitly re-checks the manifest against
#    the actual files on this machine -- build_dataloaders() would have raised
#    a KeyError already if any sample were missing from the manifest)
assert (len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset)) == (14815, 3175, 3175)
assert class_names == CLASSES, class_names
print("[1] split sizes + class order OK")

# 2) batch contract
imgs, labels, masks = next(iter(train_loader))
assert imgs.shape[1:] == (3, 224, 224), imgs.shape
assert masks.shape[1:] == (1, 224, 224), masks.shape
assert set(torch.unique(masks).tolist()) <= {0.0, 1.0}
print("[2] batch shapes OK:", imgs.shape, masks.shape)

# 3) lung-area fraction sanity over 200 random eval-transform samples.
# NOTE: measured on the real dataset (S2 verification run): mean=0.235,
# range [0.043, 0.472]. The WBS's original "~0.25-0.45" was a pre-verification
# estimate -- 0.20-0.50 below is the corrected range. This check exists to
# catch gross errors (polarity inversion ~0.99/0.01, wrong channel handling),
# not to pin down the exact anatomical distribution.
from src.datasets import JointTransform
rng = np.random.default_rng(0)
paths = [p for cls in class_names for p in sorted((Path(DATA_DIR) / cls / "images").glob("*.png"))]
sample_paths = rng.choice(paths, size=200, replace=False)
eval_tf = JointTransform(img_size=224, train=False)
fracs = []
for p in sample_paths:
    p = Path(p)
    mask_p = p.parent.parent / "masks" / p.name
    _img, m = eval_tf(Image.open(p).convert("L"), Image.open(mask_p).convert("L"))
    fracs.append(m.mean().item())
fracs = np.array(fracs)
print(f"[3] lung-area fraction: mean={fracs.mean():.3f} min={fracs.min():.3f} max={fracs.max():.3f}")
assert 0.20 < fracs.mean() < 0.50, "mask polarity or channel handling is likely wrong"

print("\nAll S2 verification assertions passed.")


### Alignment eyeball check

Overlay 8 random *augmented* train images with their masks. **Look at the output** -- do the red regions sit on the lungs after rotation/crop/flip? This is the single highest-value 30 seconds in this task: a misaligned mask silently invalidates every downstream number. (Already verified once against this exact dataset during design review -- rerun here as a live sanity check, e.g. if you're now pointing at Kaggle's copy of the dataset instead of a local one.)

In [ ]:
import matplotlib.pyplot as plt
from src.datasets import IMAGENET_MEAN, IMAGENET_STD

mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for k, ax in enumerate(axes.flat):
    show = (imgs[k] * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(show)
    ax.imshow(masks[k, 0].numpy(), cmap="Reds", alpha=0.35)
    ax.set_title(class_names[labels[k]])
    ax.axis("off")
plt.suptitle("S2 alignment check -- augmented train images (JointTransform, train=True) + masks")
plt.tight_layout()
plt.show()


## S3 -- The Lung-Region Attention module + CPU unit tests

`LungRegionAttention`, `attention_guidance_loss`, `compute_total_loss` (WBS Appendix A.2). Canonical copy lives in `src/modules/lung_attention.py`; `tests/test_lung_attention.py` must be green before any GPU work starts.

In [ ]:
from src.modules import LungRegionAttention, attention_guidance_loss, compute_total_loss

# Kaggle note: this import requires the repo to be on sys.path (see the S1 cell's
# REPO_ROOT / sys.path setup). If the repo isn't uploaded as a Kaggle Dataset,
# paste src/modules/lung_attention.py's contents into this cell instead --
# src/modules/lung_attention.py stays the single canonical copy either way
# (also reused unchanged by T23 on ResNet50).


In [ ]:
# Quick inline smoke test (pytest tests/test_lung_attention.py -v is the real suite --
# S3 DoD is 12 tests green, not this cell -- but this gives a fast visual sanity
# check directly in the notebook, useful when iterating on Kaggle).
_m = LungRegionAttention(in_channels=1024, reduction=8, gate_mode="residual")
_f = torch.randn(2, 1024, 7, 7)
_out, _att, _logits = _m(_f)
print("out:", _out.shape, "| att:", _att.shape, "| logits:", _logits.shape)
print("attention at init (should be ~0.5 everywhere, zero-init):", _att.mean().item(), _att.std().item())
print("module params:", sum(p.numel() for p in _m.parameters()), "(expect 131,329 at reduction=8, in_channels=1024)")

_mask = torch.zeros(2, 1, 224, 224); _mask[:, :, 64:160, 48:176] = 1.0
_att_loss = attention_guidance_loss(_logits, _mask)
print("attention_guidance_loss at init (uniform 0.5 attention -> should be ~log(2)=0.693):", _att_loss.item())


## S4 -- Assemble the model + parity check against vanilla

`DenseNetLungAttention` wrapper (WBS Appendix A.3), the backbone-agnostic freeze/unfreeze registry (Appendix A.4), and the CBAM comparator for arm A5 (Appendix A.4b). Parity check: gate_mode='none' must produce bit-identical logits to plain timm DenseNet121.

In [ ]:
import timm
from src.modules import build_model, freeze_backbone, unfreeze_final_blocks, print_trainable_parameters, LogitsOnly


In [ ]:
# --- Parity check: gate_mode="none" must be bit-identical to plain timm DenseNet121 ---
# (given the SAME weights -- copy the backbone state_dict across, not just same architecture)
torch.manual_seed(0)
_wrapped = build_model(num_classes=4, use_attention=True, gate_mode="none", pretrained=False)
_plain = timm.create_model("densenet121", pretrained=False, num_classes=4)
_plain.load_state_dict(_wrapped.backbone.state_dict())
_plain.eval(); _wrapped.eval()

_x = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    _plain_logits = _plain(_x)
    _wrapped_logits, _, _ = _wrapped(_x)
torch.testing.assert_close(_wrapped_logits, _plain_logits, rtol=0, atol=0)
print("PARITY CHECK PASSED: gate_mode='none' logits are bit-identical to plain timm DenseNet121")

# A0 arm (use_attention=False) must have EXACTLY the vanilla parameter count
_a0 = build_model(num_classes=4, use_attention=False, pretrained=False)
_vanilla_params = sum(p.numel() for p in _plain.parameters())
_a0_params = sum(p.numel() for p in _a0.parameters())
print(f"A0 params: {_a0_params:,} | vanilla timm params: {_vanilla_params:,}")
assert _a0_params == _vanilla_params


In [ ]:
# --- Trainable-count assertions (catch silent freeze bugs) ---
model = build_model(num_classes=4, use_attention=True, gate_mode="residual", pretrained=False)

freeze_backbone(model)
print_trainable_parameters(model, "phase1 ")
_phase1_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert _phase1_trainable == 135_429, _phase1_trainable  # classifier (4,100) + attn (131,329)

unfreeze_final_blocks(model, num_blocks=1)
print_trainable_parameters(model, "phase2 ")
_phase2_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert 2_000_000 < _phase2_trainable < 2_400_000, _phase2_trainable  # WBS estimate ~2.2M

print("\nS4 verification passed: shapes, parity, and trainable counts all match the design doc.")


## S5 -- Loss + metrics implementation

`src/modules/attention_metrics.py` (ILAR, IoU, Dice, entropy, background attention) and the Grad-CAM EIL harness (pre-gate / post-gate taps, WBS section 6.3).

Kaggle: `!pip install grad-cam` (same as Candidate C's notebook) before running the Grad-CAM cell below.

In [ ]:
from src.modules import (
    ilar, attention_iou, attention_dice, attention_entropy, background_attention,
    energy_inside_lung, cam_for, get_taps,
)


In [ ]:
# --- Metric self-check (WBS Appendix A.6, extended to all 6 functions) ---
_mask = torch.zeros(1, 1, 224, 224); _mask[:, :, 64:160, 48:176] = 1.0
_frac = _mask.mean().item()

assert abs(ilar(_mask.clone(), _mask).item() - 1.0) < 1e-4                      # perfect map
assert abs(ilar(torch.ones(1, 1, 7, 7), _mask).item() - _frac) < 0.02           # uniform map
assert ilar(1.0 - _mask, _mask).item() < 1e-4                                   # inverted map
assert abs(attention_iou(_mask.clone(), _mask).item() - 1.0) < 1e-4
assert abs(attention_dice(_mask.clone(), _mask).item() - 1.0) < 1e-4
assert abs(attention_entropy(torch.full((1, 1, 7, 7), 0.5)).item() - 0.69315) < 1e-4   # max entropy at 0.5
assert background_attention(_mask.clone(), _mask).item() < 1e-4                # attention confined to lungs -> ~0 background
assert abs(energy_inside_lung(_mask.clone(), _mask).item() - 1.0) < 1e-4       # same formula as ilar, for a CAM input

print("All 6 metric self-checks passed.")


In [ ]:
# --- Grad-CAM harness demo: arm A0's self-check (EIL_post == EIL_pre exactly, WBS 6.3) ---
_demo_model = build_model(num_classes=4, use_attention=False, pretrained=False)
_demo_images = torch.randn(2, 3, 224, 224)
_tap_post, _tap_pre = get_taps(_demo_model)

_cam_post, _preds_post = cam_for(_demo_model, _demo_images, _tap_post, torch.device("cpu"))
_cam_pre, _preds_pre = cam_for(_demo_model, _demo_images, _tap_pre, torch.device("cpu"))

_eil_post = energy_inside_lung(_cam_post, _mask.repeat(2, 1, 1, 1))
_eil_pre = energy_inside_lung(_cam_pre, _mask.repeat(2, 1, 1, 1))
print("EIL_post:", _eil_post.tolist())
print("EIL_pre: ", _eil_pre.tolist())
assert torch.allclose(_eil_post, _eil_pre, atol=1e-3), "arm A0 self-check failed: taps should be identical"
print("\nGrad-CAM harness self-check passed: arm A0's two taps agree, as expected (WBS section 6.3).")


## S6 -- Training loop

Adapted `run_epoch` / `train_phase` / `evaluate` from AuxSeg: 3-tuple batches, AMP, per-epoch val precision/recall/F1/AUROC logged to W&B, `log_summary_metrics` at the end of each phase (WBS section S6 / policy Required Information).

In [ ]:
from src.modules import run_epoch, train_phase, evaluate
from src.utils import initialize_wandb, finish_run, generate_run_name

# One function (train_phase) handles every T18 arm via the model's own
# use_attention/gate_mode flags and lambda_att -- no per-arm branching or
# copy-pasted training code (WBS S6 DoD). Which arm you get is entirely
# determined by how you build `model` (S4) and what lambda_att you pass here.


For a *real* run: `initialize_wandb(cfg, run_name=generate_run_name(cfg["model"]["name"], cfg["experiment"]["name"], cfg["experiment"]["seed"]))` before calling `train_phase(..., wandb_enabled=True)`, and `finish_run()` in a `finally` block (a crashed Kaggle session shouldn't leave a run hanging). S7's smoke test below uses `wandb_enabled=False` so it doesn't need W&B at all.

## S7 -- Smoke test + overfit test

Do not proceed to S8 until: (1) the smoke run completes end to end, (2) the 32-image overfit test reaches 100% train accuracy with att_loss falling from ~0.6931 toward the ~0.2 entropy floor (WBS section 4.4a), (3) the lambda-sensitivity spot check shows higher ILAR at lambda=5 than lambda=0.

**Do not proceed to S8 (the first expensive GPU run) until all three checks below pass.** Verified against the real dataset during design review -- results recorded in the WBS section 8/S7 and the PR description; rerun here as a live confirmation on this machine/session.

In [ ]:
# --- Check 1: smoke run (200 train / 100 val real images, 1+1 epochs) ---
import numpy as np
from src.datasets import CXRWithMaskDataset
from src.modules import build_model, freeze_backbone, unfreeze_final_blocks, train_phase, evaluate

_smoke_out = REPO_ROOT / "artifacts" / "T18_lung_attention" / "smoke_test"
_rng = np.random.default_rng(0)
_smoke_train_idx = _rng.choice(datasets["train"].indices, size=200, replace=False)
_smoke_val_idx = _rng.choice(datasets["val"].indices, size=100, replace=False)

_smoke_train_ds = CXRWithMaskDataset(datasets["train"].base_dataset, _smoke_train_idx, JointTransform(img_size=224, train=True))
_smoke_val_ds = CXRWithMaskDataset(datasets["val"].base_dataset, _smoke_val_idx, JointTransform(img_size=224, train=False))
_smoke_train_loader = torch.utils.data.DataLoader(_smoke_train_ds, batch_size=8, shuffle=True, num_workers=0)
_smoke_val_loader = torch.utils.data.DataLoader(_smoke_val_ds, batch_size=8, shuffle=False, num_workers=0)

_smoke_model = build_model(num_classes=4, use_attention=True, gate_mode="residual", pretrained=False).to(device)
freeze_backbone(_smoke_model)
_opt1 = torch.optim.AdamW([p for p in _smoke_model.parameters() if p.requires_grad], lr=1e-3)
_smoke_model = train_phase(_smoke_model, _smoke_train_loader, _smoke_val_loader, criterion, _opt1, scheduler=None,
                           device=device, epochs=1, patience=5, phase_name="phase1_frozen",
                           output_dir=_smoke_out, lambda_att=0.5, wandb_enabled=False)

unfreeze_final_blocks(_smoke_model, 1)
_opt2 = torch.optim.AdamW([p for p in _smoke_model.parameters() if p.requires_grad], lr=1e-5)
_smoke_model = train_phase(_smoke_model, _smoke_train_loader, _smoke_val_loader, criterion, _opt2, scheduler=None,
                           device=device, epochs=1, patience=5, phase_name="phase2_finetune",
                           output_dir=_smoke_out, lambda_att=0.5, wandb_enabled=False)

torch.save({"model_state_dict": _smoke_model.state_dict(), "class_names": class_names}, _smoke_out / "smoke_checkpoint.pt")
assert (_smoke_out / "smoke_checkpoint.pt").exists()

import json
for _phase in ("phase1_frozen", "phase2_finetune"):
    _hist = json.loads((_smoke_out / f"{_phase}_history.json").read_text())
    assert all(np.isfinite(row["train_loss"]) for row in _hist), f"non-finite loss in {_phase}"

_smoke_results = evaluate(_smoke_model, _smoke_val_loader, class_names, device, _smoke_out, "phase2_finetune")
assert (_smoke_out / "phase2_finetune_summary_metrics.csv").exists()

print("\nCHECK 1 PASSED: no crash, finite losses, checkpoint + history + evaluate() all wrote successfully.")
print("(Figures: deferred to S12, not yet implemented -- not part of this check.)")


In [ ]:
# --- Check 2: overfit test (32 real images, no augmentation, 100 steps, ALL params unfrozen) ---
from src.modules import compute_total_loss

_targets = np.array(datasets["train"].base_dataset.targets)
_rng2 = np.random.default_rng(1)
_overfit_idx = []
for _c in range(4):
    _class_train_idx = datasets["train"].indices[_targets[datasets["train"].indices] == _c]
    _overfit_idx.extend(_rng2.choice(_class_train_idx, size=8, replace=False))
_overfit_idx = np.array(_overfit_idx)
print("Selected 32 images, class counts:", np.bincount(_targets[_overfit_idx], minlength=4))

# Disable augmentation entirely (WBS S7: JointTransform(train=False), even for "training").
_overfit_ds = CXRWithMaskDataset(datasets["train"].base_dataset, _overfit_idx, JointTransform(img_size=224, train=False))
_overfit_loader = torch.utils.data.DataLoader(_overfit_ds, batch_size=32, shuffle=False, num_workers=0)

_of_model = build_model(num_classes=4, use_attention=True, gate_mode="residual", pretrained=False).to(device)
for p in _of_model.parameters():
    p.requires_grad = True  # ALL params unfrozen, per WBS S7
print("Trainable params:", sum(p.numel() for p in _of_model.parameters() if p.requires_grad), "(all params)")

_of_criterion = nn.CrossEntropyLoss()
_of_optimizer = torch.optim.AdamW(_of_model.parameters(), lr=1e-3)

_of_images, _of_labels, _of_masks = next(iter(_overfit_loader))
_of_images, _of_labels, _of_masks = _of_images.to(device), _of_labels.to(device), _of_masks.to(device)
if _of_masks.ndim == 3:
    _of_masks = _of_masks.unsqueeze(1)

_of_model.train()
_att_loss_history, _acc_history = [], []
for _step in range(100):
    _of_optimizer.zero_grad(set_to_none=True)
    _class_logits, _att, _att_logits = _of_model(_of_images)
    _loss, _cls_loss, _att_loss = compute_total_loss(
        _class_logits, _att_logits, _of_labels, _of_masks, _of_criterion, lambda_att=0.5, target_mode="soft"
    )
    _loss.backward()
    _of_optimizer.step()
    _preds = _class_logits.argmax(dim=1)
    _att_loss_history.append(_att_loss.item())
    _acc_history.append((_preds == _of_labels).float().mean().item())
    if _step in (0, 9, 24, 49, 74, 99):
        print(f"step {_step+1:3d}: loss={_loss.item():.4f} cls={_cls_loss.item():.4f} att={_att_loss.item():.4f} train_acc={_acc_history[-1]:.4f}")

print(f"\nFinal train accuracy: {_acc_history[-1]:.4f} (need 1.0)")
print(f"att_loss: start={_att_loss_history[0]:.4f} (expect ~0.6931) -> end={_att_loss_history[-1]:.4f} (expect falling toward ~0.2)")

assert _acc_history[-1] == 1.0, "wiring bug -- overfit accuracy did not reach 1.0"
assert abs(_att_loss_history[0] - 0.6931) < 0.05, "att_loss did not start near log(2) -- check zero-init"
assert _att_loss_history[-1] < _att_loss_history[0] - 0.2, (
    "att_loss barely moved -- attention params frozen, or passing 'a' instead of 'z' to the loss, or lambda=0"
)
assert _att_loss_history[-1] > 0.05, "att_loss went to ~0 -- target is being thresholded, not soft"
print("\nCHECK 2 PASSED.")


In [ ]:
# --- Check 3: lambda-sensitivity spot check (50 steps at lambda=0 vs lambda=5) ---
_rng3 = np.random.default_rng(2)  # different draw from check 2, same style
_lam_idx = []
for _c in range(4):
    _class_train_idx = datasets["train"].indices[_targets[datasets["train"].indices] == _c]
    _lam_idx.extend(_rng3.choice(_class_train_idx, size=8, replace=False))
_lam_idx = np.array(_lam_idx)

_lam_ds = CXRWithMaskDataset(datasets["train"].base_dataset, _lam_idx, JointTransform(img_size=224, train=False))
_lam_loader = torch.utils.data.DataLoader(_lam_ds, batch_size=32, shuffle=False, num_workers=0)
_lam_images, _lam_labels, _lam_masks = next(iter(_lam_loader))
if _lam_masks.ndim == 3:
    _lam_masks = _lam_masks.unsqueeze(1)

def _run_lambda(lam, seed=7, steps=50):
    torch.manual_seed(seed)
    m = build_model(num_classes=4, use_attention=True, gate_mode="residual", pretrained=False).to(device)
    for p in m.parameters():
        p.requires_grad = True
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3)
    m.train()
    for _ in range(steps):
        opt.zero_grad(set_to_none=True)
        cls_logits, att, att_logits = m(_lam_images)
        loss, _cls, _att = compute_total_loss(cls_logits, att_logits, _lam_labels, _lam_masks, crit, lambda_att=lam, target_mode="soft")
        loss.backward()
        opt.step()
    m.eval()
    with torch.no_grad():
        _, att_final, _ = m(_lam_images)
        return ilar(att_final.float(), _lam_masks).mean().item()

print("Running lambda=0 and lambda=5 (same seed for both, so the only difference is lambda)...")
_ilar_lam0 = _run_lambda(0.0)
_ilar_lam5 = _run_lambda(5.0)
print(f"ILAR: lambda=0 -> {_ilar_lam0:.4f} | lambda=5 -> {_ilar_lam5:.4f} | delta={_ilar_lam5 - _ilar_lam0:.4f}")

assert _ilar_lam5 > _ilar_lam0 + 0.05, "guidance signal is not reaching the module -- ILAR did not rise meaningfully"
print("\nCHECK 3 PASSED.")
print("\n" + "="*70)
print("ALL S7 CHECKS PASSED -- clear to proceed to S8.")
print("="*70)


## S8 -- Arm A0 -- the vanilla control (full schedule)

`use_attention=False`. Sanity gate: judge against the measured 90.5-92% macro-F1 range (WBS section 2.4), not a single point estimate from the possibly-placeholder proposal table.

**Run this on Kaggle (GPU), not locally.** Estimated 2.5-4h on a T4. Before running: (1) W&B key set up via Kaggle Secrets, per docs/wandb_setup.md and S0 -- initialize_wandb() below will try to authenticate online; (2) confirm T13's config if it landed, else this uses section 3.4a's defaults already baked into configs/densenet121_lung_attention.yaml. Every piece this cell composes (build_model / freeze_backbone / unfreeze_final_blocks / train_phase / evaluate) was individually verified in S3-S6, and the exact same freeze-train-unfreeze-train-evaluate-save sequence already ran for real in S7 check 1's smoke test -- what has NOT been run is the real 15+40 epoch schedule itself, which only makes sense on a GPU.

In [ ]:
from src.modules import run_full_arm

a0_cfg = merge_overrides(cfg, {
    "module.use_attention": False,
    "module.lambda_att": 0.0,
    "experiment.name": "T18-A0-vanilla",
})

A0_OUT = REPO_ROOT / "artifacts" / "T18_lung_attention" / "runs" / "A0_vanilla"

a0_results = run_full_arm(
    "A0_vanilla", a0_cfg, train_loader, val_loader, test_loader, class_names, criterion,
    device, A0_OUT, backbone_name="densenet121", wandb_enabled=True,
)
print(f"\nSaved checkpoint to {A0_OUT / 'densenet121_A0_vanilla.pt'}")


In [ ]:
# --- S8 sanity gate (WBS section 2.4 / S8) ---
macro_f1 = a0_results["classification_report"]["macro avg"]["f1-score"]
print(f"A0 test macro-F1: {macro_f1:.4f}")

if 0.905 <= macro_f1 <= 0.92:
    print("Sanity gate: PASS -- in the expected 90.5-92% range. Proceed to S9.")
elif 0.89 <= macro_f1 < 0.905:
    print("Sanity gate: BORDERLINE -- check you\'re on the section 3.4a config "
          "(phase1_lr=3e-4, phase2_lr=3e-5, weight_decay=1e-3), not the guide\'s untuned defaults. "
          "Landing near AuxSeg\'s 90.22% is NOT evidence you matched the baseline.")
elif macro_f1 < 0.85:
    print("Sanity gate: FAIL -- likely a pipeline bug (roughly frozen-phase-1-only level). "
          "Stop and debug before proceeding to S9.")
else:
    print("Sanity gate: outside the pre-registered bands -- inspect manually before proceeding.")


## S9 -- Lambda sweep (short schedule) + pre-registered selection

Mirrors T16's TUNING_CONFIGS -> tuning_summary.csv -> selected_config.json pattern exactly (WBS section S9). Test set stays out of scope for this entire section.

**Run this on Kaggle (GPU), not locally.** 5 short-schedule configs (4+6 epochs each) -- estimated 3-4h total on a T4. Mirrors T16's exact machinery (kusal-notebooks/efficientnet-b0-baseline-model.ipynb, cells 26-28): TUNING_CONFIGS -> per-config dir -> tuning_summary.csv -> selected_config.json -> (S10's) one full final run. **Validation only below -- test_loader is never referenced in any S9 cell.** A single peek at test performance during selection invalidates the protocol (WBS section R13).

In [ ]:
import pandas as pd
from src.modules import build_optimizer, build_scheduler, best_history_row

SWEEP_ROOT = REPO_ROOT / "artifacts" / "T18_lung_attention" / "sweep"
SWEEP_ROOT.mkdir(parents=True, exist_ok=True)
SEED = cfg["experiment"]["seed"]

# Short schedule for ranking only -- WBS section S9. The full schedule
# (S8's 15+40) is reserved for arms A1/A4/A2/A5 in S10, using lambda*
# selected here.
TUNE_PHASE1_EPOCHS = 4
TUNE_PHASE2_EPOCHS = 6
TUNE_PATIENCE = 3

# BASE is derived from the actual config (section 3.4a's values, or T13's
# if it lands later) -- not hardcoded, so updating the YAML is enough.
BASE = {
    "phase1_lr": cfg["training"]["phase1_lr"],
    "phase2_lr": cfg["training"]["phase2_lr"],
    "weight_decay": cfg["training"]["weight_decay"],
    "optimizer": cfg["training"]["optimizer"],
    "scheduler": cfg["scheduler"]["name"],
    "gate_mode": cfg["module"]["gate_mode"],
}

TUNING_CONFIGS = [
    {"name": "lam_0.0", "lambda_att": 0.0, **BASE},   # == arm A1's short-schedule reference row
    {"name": "lam_0.1", "lambda_att": 0.1, **BASE},
    {"name": "lam_0.3", "lambda_att": 0.3, **BASE},
    {"name": "lam_0.5", "lambda_att": 0.5, **BASE},
    {"name": "lam_1.0", "lambda_att": 1.0, **BASE},
]
pd.DataFrame(TUNING_CONFIGS)


In [ ]:
# --- Validation-only targeted sweep (WBS section S9) ---
from src.utils import set_seed, initialize_wandb, generate_run_name, finish_run

tuning_results = []

for sweep_cfg in TUNING_CONFIGS:
    print("\n" + "=" * 90)
    print(f"SWEEP: {sweep_cfg['name']} | {sweep_cfg}")
    print("=" * 90)

    set_seed(SEED)  # reset RNG per config -- T16's convention, every config starts comparably
    cfg_dir = SWEEP_ROOT / sweep_cfg["name"]
    cfg_dir.mkdir(parents=True, exist_ok=True)

    run = initialize_wandb(
        cfg, run_name=generate_run_name("densenet121", f"T18-sweep-{sweep_cfg['name']}", SEED),
    )
    try:
        sweep_model = build_model(
            num_classes=cfg["model"]["num_classes"], use_attention=True,
            gate_mode=sweep_cfg["gate_mode"], pretrained=cfg["model"]["pretrained"],
        ).to(device)

        freeze_backbone(sweep_model)
        opt1 = build_optimizer(sweep_model, sweep_cfg["optimizer"], sweep_cfg["phase1_lr"], sweep_cfg["weight_decay"])
        sched1 = build_scheduler(opt1, sweep_cfg["scheduler"], TUNE_PHASE1_EPOCHS)
        sweep_model = train_phase(
            sweep_model, train_loader, val_loader, criterion, opt1, sched1, device,
            epochs=TUNE_PHASE1_EPOCHS, patience=TUNE_PATIENCE, phase_name="phase1_frozen",
            output_dir=cfg_dir, lambda_att=sweep_cfg["lambda_att"], wandb_enabled=True,
        )
        # NO evaluate() call here -- no test_loader in scope for this cell at all (WBS R13).

        unfreeze_final_blocks(sweep_model, cfg["training"]["unfreeze_blocks"])
        opt2 = build_optimizer(sweep_model, sweep_cfg["optimizer"], sweep_cfg["phase2_lr"], sweep_cfg["weight_decay"])
        sched2 = build_scheduler(opt2, sweep_cfg["scheduler"], TUNE_PHASE2_EPOCHS)
        sweep_model = train_phase(
            sweep_model, train_loader, val_loader, criterion, opt2, sched2, device,
            epochs=TUNE_PHASE2_EPOCHS, patience=TUNE_PATIENCE, phase_name="phase2_finetune",
            output_dir=cfg_dir, lambda_att=sweep_cfg["lambda_att"], wandb_enabled=True,
        )

        best = best_history_row(cfg_dir / "phase2_finetune_history.json")
        tuning_results.append({
            **sweep_cfg,
            "best_epoch": best["epoch"],
            "best_val_loss": best["val_loss"],
            "val_acc_at_best_loss": best["val_acc"],
            "val_macro_f1_at_best": best["val_f1"],
            "val_ilar_at_best": best["val_ilar"],
        })
    finally:
        finish_run()

    # WBS S9 pitfall: never carry a "final" model out of the sweep loop --
    # S10 rebuilds fresh from selected_config.json instead of reusing this.
    del sweep_model, opt1, opt2, sched1, sched2
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nSweep complete.")


In [ ]:
# --- Write tuning_summary.csv (WBS section S9, column order as specified) ---
tuning_df = pd.DataFrame(tuning_results)[[
    "name", "lambda_att", "phase1_lr", "phase2_lr", "weight_decay", "optimizer", "scheduler",
    "best_epoch", "best_val_loss", "val_acc_at_best_loss", "val_macro_f1_at_best", "val_ilar_at_best",
]]
tuning_df.to_csv(SWEEP_ROOT / "tuning_summary.csv", index=False)
print("=== SWEEP SUMMARY ===")
tuning_df


In [ ]:
# --- lambda_sweep.png: accuracy/faithfulness trade-off (WBS section S9 step 4) ---
import matplotlib.pyplot as plt

plot_df = tuning_df.sort_values("lambda_att")
fig, ax1 = plt.subplots(figsize=(8, 5))
ax2 = ax1.twinx()

ax1.plot(plot_df["lambda_att"], plot_df["val_macro_f1_at_best"], "o-", color="tab:blue", label="val macro-F1")
ax2.plot(plot_df["lambda_att"], plot_df["val_ilar_at_best"], "s-", color="tab:red", label="val ILAR")

ax1.set_xlabel("lambda_att")
ax1.set_ylabel("val macro-F1", color="tab:blue")
ax2.set_ylabel("val ILAR", color="tab:red")
ax1.tick_params(axis="y", labelcolor="tab:blue")
ax2.tick_params(axis="y", labelcolor="tab:red")
plt.title("T18 lambda sweep: accuracy/faithfulness trade-off (short schedule)")
fig.tight_layout()

FIG_DIR = REPO_ROOT / "artifacts" / "T18_lung_attention" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG_DIR / "lambda_sweep.png", dpi=150)
plt.show()


In [ ]:
# --- Apply the pre-registered selection rule (WBS section 5.3) -> lambda* ---
TOLERANCE = 0.005  # 0.5 percentage points, absolute

reference_row = tuning_df[tuning_df["name"] == "lam_0.0"].iloc[0]
reference_f1 = reference_row["val_macro_f1_at_best"]
print(f"Reference (lam_0.0) val macro-F1: {reference_f1:.4f}")

candidates = tuning_df[tuning_df["val_macro_f1_at_best"] >= reference_f1 - TOLERANCE].copy()
if candidates.empty:
    raise RuntimeError(
        "No lambda satisfies the 0.5pp tolerance -- per WBS section 5.3, extend the grid "
        "downward (e.g. lambda in {0.05, 0.03}) rather than relaxing the tolerance."
    )

# Largest lambda within tolerance; ties (won't occur with 5 distinct lambda
# values, but implemented per the written rule) broken by higher val ILAR.
max_lambda = candidates["lambda_att"].max()
tied = candidates[candidates["lambda_att"] == max_lambda].sort_values("val_ilar_at_best", ascending=False)
winner_row = tied.iloc[0]

winner = {
    "name": winner_row["name"],
    "lambda_att": float(winner_row["lambda_att"]),
    "phase1_lr": float(winner_row["phase1_lr"]),
    "phase2_lr": float(winner_row["phase2_lr"]),
    "weight_decay": float(winner_row["weight_decay"]),
    "optimizer": winner_row["optimizer"],
    "scheduler": winner_row["scheduler"],
    "gate_mode": winner_row["gate_mode"],
    "target_mode": cfg["module"]["target_mode"],
    "reduction": cfg["module"]["reduction"],
    "rule": "largest lambda whose val macro-F1 is within 0.5pp of lam_0.0; ties -> higher val ILAR",
    "reference_row": "lam_0.0",
    "reference_val_macro_f1": float(reference_f1),
    "schedule": f"short({TUNE_PHASE1_EPOCHS}+{TUNE_PHASE2_EPOCHS})",
    "seed": SEED,
    # Update this if T13's tuned config supersedes T16's winner (section 3.4a) later.
    "config_provenance": "T16 winner D_more_wd (T13 unavailable at time of writing)",
}

with open(SWEEP_ROOT / "selected_config.json", "w") as f:
    json.dump(winner, f, indent=2)

print("Selected configuration (lambda*):")
print(json.dumps(winner, indent=2))


## S10 -- Full-schedule runs -- A1, A4, A2, A5 (and optionally A3)

Priority order: A2 (headline) -> A1 (isolates supervision) -> A5 (CBAM, required by the assignment brief section 9) -> A4 (isolates gating) -> A3 (optional). Re-seed at the start of every arm.

**Run this on Kaggle (GPU), not locally.** 4 full-schedule arms (15+40 epochs each, priority order below) -- estimated 11-16h across sessions on a T4. Requires S9's `selected_config.json` to already exist (lambda*). Re-seeds fresh at the start of every arm (`run_full_arm` calls `set_seed()` internally) -- WBS section S10's pitfall: continuing a session without re-seeding makes arm N's init depend on how many random numbers arm N-1 consumed, a real confound. If a session times out partway through, the arms already completed keep their artifacts; just rerun the remaining priority-ordered cells.

In [ ]:
# --- Load lambda* from S9's sweep ---
SELECTED_CONFIG_PATH = SWEEP_ROOT / "selected_config.json"
assert SELECTED_CONFIG_PATH.exists(), (
    f"{SELECTED_CONFIG_PATH} not found -- run S9's sweep first, it must produce this file."
)
with open(SELECTED_CONFIG_PATH) as f:
    selected = json.load(f)
LAMBDA_STAR = selected["lambda_att"]
print(f"Using lambda* = {LAMBDA_STAR} (selected by S9, rule: {selected['rule']})")

S10_RUNS_ROOT = REPO_ROOT / "artifacts" / "T18_lung_attention" / "runs"

# Arm -> flags mapping (WBS Appendix A.3's table). Every arm shares cfg's
# base training/scheduler/model settings; only the module block differs.
ARM_CONFIGS = {
    "A2_full": merge_overrides(cfg, {
        "module.use_attention": True, "module.attention": "lung",
        "module.gate_mode": "residual", "module.lambda_att": LAMBDA_STAR,
        "experiment.name": "T18-A2-full",
    }),
    "A1_gate_only": merge_overrides(cfg, {
        "module.use_attention": True, "module.attention": "lung",
        "module.gate_mode": "residual", "module.lambda_att": 0.0,
        "experiment.name": "T18-A1-gate-only",
    }),
    "A5_cbam": merge_overrides(cfg, {
        "module.use_attention": True, "module.attention": "cbam",
        "module.lambda_att": 0.0,  # CBAM is never mask-supervised (WBS Appendix A.4b)
        "experiment.name": "T18-A5-cbam",
    }),
    "A4_guidance_only": merge_overrides(cfg, {
        "module.use_attention": True, "module.attention": "lung",
        "module.gate_mode": "none", "module.lambda_att": LAMBDA_STAR,
        "experiment.name": "T18-A4-guidance-only",
    }),
    "A3_multiply": merge_overrides(cfg, {  # optional -- drop first if GPU budget is tight
        "module.use_attention": True, "module.attention": "lung",
        "module.gate_mode": "multiply", "module.lambda_att": LAMBDA_STAR,
        "experiment.name": "T18-A3-multiply",
    }),
}

# Priority order (WBS section S10 table) -- run top to bottom; if a
# session runs out of budget, everything below the last completed arm
# is what's still missing, in the order it's safest to drop.
ARM_PRIORITY = ["A2_full", "A1_gate_only", "A5_cbam", "A4_guidance_only", "A3_multiply"]


In [ ]:
# --- Run each arm in priority order (WBS section S10) ---
from src.modules import run_full_arm

arm_results = {}
for arm_name in ARM_PRIORITY:
    print("\n" + "#" * 90)
    print(f"# ARM: {arm_name}")
    print("#" * 90)
    arm_out = S10_RUNS_ROOT / arm_name
    arm_results[arm_name] = run_full_arm(
        arm_name, ARM_CONFIGS[arm_name], train_loader, val_loader, test_loader,
        class_names, criterion, device, arm_out,
        backbone_name="densenet121", wandb_enabled=True,
    )
    print(f"Saved: {arm_out / f'densenet121_{arm_name}.pt'}")

print("\nAll S10 arms complete.")


In [ ]:
# --- S10 DoD check ---
for arm_name in ["A0_vanilla", "A1_gate_only", "A4_guidance_only", "A2_full"]:
    arm_dir = S10_RUNS_ROOT / arm_name
    results_path = arm_dir / "phase2_finetune_test_results.json"
    ckpt_path = arm_dir / f"densenet121_{arm_name}.pt"
    ok = results_path.exists() and ckpt_path.exists()
    print(f"{arm_name}: {'OK' if ok else 'MISSING'} "
          f"(results={results_path.exists()}, checkpoint={ckpt_path.exists()})")


## S11 -- Evaluation, faithfulness, comparison table

Grad-CAM EIL at both taps, attention metrics, `T18_comparison_table.csv`, explicit pass/fail verdicts against acceptance criteria A1-A6, per-image prediction CSVs for Member 3's statistics (T37).

**Run this on Kaggle (GPU), not locally.** Grad-CAM over the fixed subsample x every arm x 2 taps -- estimated ~1h on a T4. Requires S8/S10's checkpoints to already exist (`densenet121_A0_vanilla.pt`, `..._A1_gate_only.pt`, `..._A4_guidance_only.pt`, `..._A2_full.pt`, and A5/A3 if run).

In [ ]:
# --- Fixed, stratified Grad-CAM subsample -- same images for every arm (WBS S11 budget note) ---
from src.modules import stratified_cam_subset, build_per_image_predictions, build_comparison_table, check_acceptance_criteria

CAM_N = 1000  # drop to e.g. 300 if a session is running short on time; the SET must stay fixed across arms either way
cam_subset = stratified_cam_subset(datasets["test"], n=CAM_N, seed=cfg["experiment"]["seed"])
print(f"Grad-CAM subsample: {len(cam_subset)} / {len(datasets['test'])} test images")

S11_OUT = REPO_ROOT / "artifacts" / "T18_lung_attention"
S11_OUT.mkdir(parents=True, exist_ok=True)
with open(S11_OUT / "cam_subset_indices.json", "w") as f:
    json.dump({"n": len(cam_subset), "seed": cfg["experiment"]["seed"], "positions": cam_subset.tolist()}, f, indent=2)


In [ ]:
# --- Helper: rebuild a model from one of S8/S10's saved checkpoints ---
def load_arm_model(ckpt_path, backbone_name="densenet121"):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    arm_cfg = ckpt["config"]
    m = arm_cfg["module"]
    model = build_model(
        num_classes=arm_cfg["model"]["num_classes"],
        use_attention=m["use_attention"], attention=m.get("attention", "lung"),
        gate_mode=m.get("gate_mode", "residual"), reduction=m.get("reduction", 8),
        backbone_name=backbone_name, pretrained=False,  # weights come from the checkpoint, not ImageNet
    ).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    return model, arm_cfg, ckpt.get("class_names", class_names)

# Arm name -> (checkpoint path, output dir it was trained into).
ARM_CHECKPOINTS = {
    "A0_vanilla": A0_OUT / "densenet121_A0_vanilla.pt",
    "A1_gate_only": S10_RUNS_ROOT / "A1_gate_only" / "densenet121_A1_gate_only.pt",
    "A4_guidance_only": S10_RUNS_ROOT / "A4_guidance_only" / "densenet121_A4_guidance_only.pt",
    "A2_full": S10_RUNS_ROOT / "A2_full" / "densenet121_A2_full.pt",
    # A5/A3 optional -- add here if you ran them:
    # "A5_cbam": S10_RUNS_ROOT / "A5_cbam" / "densenet121_A5_cbam.pt",
    # "A3_multiply": S10_RUNS_ROOT / "A3_multiply" / "densenet121_A3_multiply.pt",
}


In [ ]:
# --- Per-arm: per-image predictions + Grad-CAM EIL on the fixed subsample (WBS S11 step 1) ---
arms_for_table = {}

for arm_name, ckpt_path in ARM_CHECKPOINTS.items():
    if not ckpt_path.exists():
        print(f"SKIPPING {arm_name}: checkpoint not found at {ckpt_path} (not run yet?)")
        continue

    print(f"\n=== {arm_name} ===")
    model, arm_cfg, arm_class_names = load_arm_model(ckpt_path)

    arm_dir = ckpt_path.parent
    results_path = arm_dir / "phase2_finetune_test_results.json"
    assert results_path.exists(), f"{results_path} missing -- evaluate() should have written this during training"
    with open(results_path) as f:
        evaluate_results = json.load(f)

    per_image_df = build_per_image_predictions(
        model, datasets["test"], arm_class_names, device, cam_subset=cam_subset,
    )
    per_image_df.to_csv(arm_dir / "per_image_predictions.csv", index=False)

    attention_metrics_summary = {
        "attention_ilar": evaluate_results.get("attention_ilar"),
        "attention_dice": evaluate_results.get("attention_dice"),
        "attention_iou": evaluate_results.get("attention_iou"),
        "cam_eil_post_mean": float(per_image_df["eil_post"].dropna().mean()) if per_image_df["eil_post"].notna().any() else None,
        "cam_eil_pre_mean": float(per_image_df["eil_pre"].dropna().mean()) if per_image_df["eil_pre"].notna().any() else None,
        "cam_subset_n": len(cam_subset),
    }
    with open(arm_dir / "attention_metrics.json", "w") as f:
        json.dump(attention_metrics_summary, f, indent=2)
    print(f"  attention_metrics.json: {attention_metrics_summary}")

    arms_for_table[arm_name] = {
        "gate_mode": arm_cfg["module"].get("gate_mode") if arm_cfg["module"]["use_attention"] else "none",
        "lambda_att": arm_cfg["module"]["lambda_att"],
        "evaluate_results": evaluate_results,
        "per_image_df": per_image_df,
    }

print(f"\n{len(arms_for_table)} / {len(ARM_CHECKPOINTS)} arms processed.")


In [ ]:
# --- T18_comparison_table.csv (WBS S11 step 3) ---
comparison_df = build_comparison_table(
    arms_for_table, reference_arm="A0_vanilla" if "A0_vanilla" in arms_for_table else None,
    output_csv=S11_OUT / "T18_comparison_table.csv",
)
comparison_df


In [ ]:
# --- Acceptance criteria A1-A4 (WBS section 1.3), checked explicitly, not by eye ---
if "A2_full" in arms_for_table:
    verdicts = check_acceptance_criteria(comparison_df, headline_arm="A2_full")
    for name, v in verdicts.items():
        status = "PASS" if v["pass"] else "FAIL"
        print(f"[{status}] {name}: {v['value']:.4f} (target: {v['target']})")
    with open(S11_OUT / "acceptance_criteria_A1_A4.json", "w") as f:
        json.dump(verdicts, f, indent=2)
    print("\nA failing verdict is a legitimate finding to report (WBS section 1.3's note),")
    print("not a reason to keep tuning until it passes. Write these verdicts into the module card (S15).")
else:
    print("A2_full not in arms_for_table -- run S10's A2 arm first.")


## S12 -- Figures

Heat-map overlays (3 per class), the attention grid, the lambda-sweep trade-off plot.

**Run this on Kaggle (GPU), not locally.** Needs S11's arms_for_table already populated (A0_vanilla and A2_full at minimum) -- Grad-CAM per selected image, so this is fast once S11 has run (a handful of images, not the full subsample). `lambda_sweep.png` (step 3) was already produced in S9 -- nothing here duplicates it. Step 4 (attention evolving over epochs) was decided against -- see the design doc's note on this being collectible only during training, not after.

In [ ]:
# --- S12 steps 1-2: select images, render heat-map overlays + attention grid ---
from src.modules import select_heatmap_images, plot_heatmap_row, plot_attention_grid, cam_for, get_taps
from src.datasets import IMAGENET_MEAN, IMAGENET_STD
import torch.nn.functional as F

assert "A0_vanilla" in arms_for_table and "A2_full" in arms_for_table, (
    "S12 needs both A0 and A2's per-image predictions from S11 -- run S11 first."
)

FIG_DIR = REPO_ROOT / "artifacts" / "T18_lung_attention" / "figures"
HEATMAP_DIR = FIG_DIR / "heatmaps"
HEATMAP_DIR.mkdir(parents=True, exist_ok=True)

_mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
_std = torch.tensor(IMAGENET_STD).view(3, 1, 1)


def _dataset_index_for_path(test_dataset, image_path):
    for i in range(len(test_dataset)):
        p = str(test_dataset.base_dataset.samples[test_dataset.indices[i]][0])
        if p == image_path:
            return i
    raise ValueError(f"image_path not found in test dataset: {image_path}")


def _denormalize_for_display(image_tensor):
    # JointTransform's eval path is grayscale-replicated to 3 channels
    # (all 3 identical), so any single channel is the display image.
    return (image_tensor * _std + _mean).clamp(0, 1)[0].numpy()


In [ ]:
# --- Step 1: 3 images/class x 4 classes -> heat-map overlay rows ---
picks = select_heatmap_images(
    arms_for_table["A0_vanilla"]["per_image_df"], arms_for_table["A2_full"]["per_image_df"],
    class_names, seed=cfg["experiment"]["seed"],
)
print(f"Selected {len(picks)} images ({picks['reason'].value_counts().to_dict()})")
print("NOTE: verify the 'shortcut_artifact_proxy' picks visually -- ilar is a proxy for")
print("'attention on background', not a detector for text markers/devices/borders specifically.")

model_a0, _a0_cfg, _ = load_arm_model(ARM_CHECKPOINTS["A0_vanilla"])
model_a2, _a2_cfg, _ = load_arm_model(ARM_CHECKPOINTS["A2_full"])
tap_a0_post, _tap_a0_pre = get_taps(model_a0)
tap_a2_post, _tap_a2_pre = get_taps(model_a2)

for _, row in picks.iterrows():
    idx = _dataset_index_for_path(datasets["test"], row["image_path"])
    image_t, _label_t, mask_t = datasets["test"][idx]
    image_batch = image_t.unsqueeze(0)
    display_img = _denormalize_for_display(image_t)
    display_mask = mask_t.squeeze().numpy()

    a0_cam, _ = cam_for(model_a0, image_batch, tap_a0_post, device)
    with torch.no_grad():
        _, a2_att, _ = model_a2(image_batch.to(device))
    a2_att_up = F.interpolate(a2_att.float(), size=(224, 224), mode="bilinear", align_corners=False)
    a2_cam, _ = cam_for(model_a2, image_batch, tap_a2_post, device)

    safe_class = row["class_name"].replace(" ", "_")
    out_path = HEATMAP_DIR / f"{safe_class}_{idx}_{row['reason']}.png"
    plot_heatmap_row(
        display_img, display_mask,
        a0_cam[0, 0].detach().cpu().numpy(),
        a2_att_up[0, 0].detach().cpu().numpy(),
        a2_cam[0, 0].detach().cpu().numpy(),
        f"{row['class_name']} -- {row['reason']}", out_path,
    )
    print(f"Saved {out_path}")


In [ ]:
# --- Step 2: attention_grid.png -- 4 classes x 4 samples of A2's attention (WBS S12 step 2) ---
rng_grid = np.random.default_rng(cfg["experiment"]["seed"])
grid_images, grid_attentions, grid_labels = [], [], []

test_targets = np.array([datasets["test"].base_dataset.targets[i] for i in datasets["test"].indices])
for class_idx, cname in enumerate(class_names):
    class_positions = np.where(test_targets == class_idx)[0]
    chosen = rng_grid.choice(class_positions, size=min(4, len(class_positions)), replace=False)
    for pos in chosen:
        image_t, _label_t, _mask_t = datasets["test"][int(pos)]
        with torch.no_grad():
            _, att, _ = model_a2(image_t.unsqueeze(0).to(device))
        att_up = F.interpolate(att.float(), size=(224, 224), mode="bilinear", align_corners=False)
        grid_images.append(_denormalize_for_display(image_t))
        grid_attentions.append(att_up[0, 0].detach().cpu().numpy())
        grid_labels.append(cname)

plot_attention_grid(grid_images, grid_attentions, grid_labels, class_names, FIG_DIR / "attention_grid.png", n_per_class=4)
print(f"Saved {FIG_DIR / 'attention_grid.png'}")


In [ ]:
# --- S12 DoD check ---
n_heatmaps = len(list(HEATMAP_DIR.glob('*.png')))
has_grid = (FIG_DIR / "attention_grid.png").exists()
has_sweep = (FIG_DIR / "lambda_sweep.png").exists()  # from S9
print(f"Heat-map overlays: {n_heatmaps} (need >= 12)")
print(f"attention_grid.png: {'OK' if has_grid else 'MISSING'}")
print(f"lambda_sweep.png (S9): {'OK' if has_sweep else 'MISSING'}")
assert n_heatmaps >= 12 and has_grid and has_sweep, "S12 DoD not met -- see missing items above"
print("\nS12 DoD met. Reference these files by name in the module card (S15).")


## S13 -- Efficiency measurement (A6)

`kusal-notebooks/efficiency.py::benchmark_model`, wrapped in `LogitsOnly` -- verify against the analytic estimate (+131,329 params, +0.22% GFLOPs).

## S14 -- Multi-seed repeat (optional, budget permitting)

A0 and A2 only, seeds 123 and 2026, full schedule. Explicitly state single-seed in the module card if this doesn't run.

## S15 -- Package and hand off

`T18_module_card.md`, deliverables to Member 3 / Member 2 / Member 5, the `LogitsOnly` wrapper note, RSNA class-mapping note, contribution list for the colour-highlighted paper.

## S16 -- Paper material

~400 words Methods, ~250 words Results -- draft in WBS Appendix B, edit against the actual `T18_comparison_table.csv` numbers before handing off.